# 7.6 - Observability with Langfuse

**Module 7 - LLM API Engineering** | Netsetos GenAI Engineering

This notebook builds an LLM observability layer by hand: it traces every model call (input, output, tokens, cost, latency), scores whether the answer was actually good, rolls thousands of traces into the five metrics teams watch, and ends with a single chat class where every message is traced, auto-scored, costed, and alertable. Along the way it maps the tool landscape (Langfuse, LangSmith, Helicone, OTEL), the eval frameworks (RAGAS, DeepEval), and the DPDP-compliant India stack.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the SDKs this lesson uses: the observability backends (Langfuse, LangSmith), a model provider client, and env-file loading. On Colab or a fresh machine, uncomment the line and run it once; if the packages are already present you can skip it.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install openai "langfuse>=3" langsmith google-genai python-dotenv -q  # noqa


**Explanation**

A one-line dependency install for the whole notebook - not a model call, just environment prep.

**How the code works - step by step**
- **`openai`** - OpenAI client, used later only to show the Helicone proxy pattern.
- **`langfuse>=3`** - the primary observability SDK (v3 is OpenTelemetry-based); pinned to v3 because the whole notebook uses the v3 context-manager and `@observe` API.
- **`langsmith`** - LangChain's observability tool, for the second-vendor demo.
- **`google-genai`** - the Gemini client that makes the actual LLM calls throughout.
- **`python-dotenv`** - loads keys from a `.env` file so nothing is hardcoded.

**In one line:** install the two observability backends + the Gemini client + dotenv.

**What you'll see:** (no output - environment prep)

## Setup - API keys

Pull provider credentials from the environment instead of hardcoding them. Any one provider key is enough to start the demos; the Gemini calls in this notebook read `GOOGLE_API_KEY`, and the Langfuse calls additionally need `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` in the environment.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`) for every live model call, plus Langfuse keys for the tracing cells.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Key-loading boilerplate that sets empty defaults so imports don't crash - it reads from the environment, never storing real secrets in the file.

**How the code works - step by step**
- **`os.environ.setdefault("OPENAI_API_KEY", "")`** - registers an empty OpenAI key only if one isn't already set, so the variable exists.
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - same for the Gemini key that the `genai.Client()` calls will read.
- The inline comments point at where to get each key (platform.openai.com, aistudio.google.com).

**In one line:** make sure the key variables exist without ever hardcoding a secret.

**What you'll see:** (no output - environment prep)

## 1 - The outage nobody could explain

Before any tooling, feel the problem. This cell is a reconstructed incident channel from a real failure mode: an AI support bot silently got worse, and every systems metric stayed green. It frames why LLM observability exists.

In [ ]:
# Output: the incident channel, reconstructed
#
# 14:02  "answers are wrong on billing questions"   (3 users)
# 14:20  on-call checks: latency NORMAL, error rate NORMAL, uptime 100%
# 14:41  "still bad" (12 users). on-call: "everything is green?!"
# 15:30  guessing. rolls back a deploy. no change. rolls it forward. no change.
# 18:10  someone asks: "what PROMPT are we actually sending now?"
#        ...nobody knows. there is no record of any single request.
#
# Root cause (found next day, by luck): the provider silently updated the
# model behind the same name. Quality dropped. Latency and errors never moved.
# Four hours of guessing that ONE traced request would have ended in minutes.

**Explanation**

A comment-only cell - there is no executable code, just a narrated incident log. It exists to make the missing layer concrete: latency, error rate, and uptime were all normal, yet the answers were wrong.

**How the code works - step by step**
- The `# Output:` block is a fake timeline: complaints arrive at 14:02, on-call confirms latency/errors/uptime are all NORMAL, more complaints pile up, someone rolls a deploy back and forward with no effect.
- The turning point (18:10): nobody can answer 'what PROMPT are we actually sending now?' because there is no record of any single request.
- The root cause noted at the bottom: the provider silently updated the model behind the same name - quality dropped while latency and errors never moved.

**In one line:** every metric they had was a systems metric; none could see the content or quality of a call.

**What you'll see:** No output - the cell is entirely comments. Running it does nothing; it's the story that motivates the rest of the notebook.

## 2 - Langfuse: trace your first LLM call

Start with the primitive everything else builds on. Wrap one Gemini call in a Langfuse *generation* so the dashboard records its input, output, token counts, cost, and latency - the content-and-quality layer the cold-open team was missing.

> **Needs a Gemini API key** (`GOOGLE_API_KEY`) and Langfuse keys (`LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY`).

In [ ]:
# =============================================
# LANGFUSE: instrument your first LLM call (v3 SDK)
# pip install "langfuse>=3" google-genai
# Sign up free: https://cloud.langfuse.com  (ClickHouse acquired
# Langfuse in Jan 2026; it stays open-source and self-hostable.)
# =============================================

from langfuse import get_client
from google import genai
import os, time

# Reads LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY / LANGFUSE_HOST from env.
langfuse = get_client()
client = genai.Client()          # reads GOOGLE_API_KEY
MODEL = "gemini-2.5-flash"       # gemini-2.0-flash was shut down 2026-06-01

def ask_with_tracing(question: str, user_id: str = "anonymous") -> str:
    """Make an LLM call wrapped in a Langfuse generation (v3 context managers)."""
    # A generation = an LLM call. The context manager opens/closes the span
    # and makes it the CURRENT observation for anything nested inside.
    with langfuse.start_as_current_generation(
        name="gemini-call",
        model=MODEL,
        input=question,
        model_parameters={"temperature": 0.3},
    ) as generation:
        start = time.time()
        resp = client.models.generate_content(model=MODEL, contents=question)
        answer = resp.text

        # Attach the real numbers Langfuse costs and charts on:
        generation.update(
            output=answer,
            usage_details={
                "input_tokens": resp.usage_metadata.prompt_token_count,
                "output_tokens": resp.usage_metadata.candidates_token_count,
            },
            metadata={"latency_ms": round((time.time() - start) * 1000)},
        )
        # Trace-level metadata (who/where) belongs on the enclosing trace:
        langfuse.update_current_trace(user_id=user_id, metadata={"source": "web"})
    return answer

answer = ask_with_tracing("What is RAG in 2 sentences?", user_id="student_001")
print(f"Answer: {answer}")
langfuse.flush()   # short-lived script: force-send before exit

# Output:
#   Answer: RAG (Retrieval-Augmented Generation) retrieves relevant documents
#   first, then feeds them to the LLM so answers are grounded in real data...
#   -> a "gemini-call" generation now appears in your Langfuse dashboard with
#      input, output, 41 input / 58 output tokens, cost, and latency.


**Explanation**

The manual-tracing pattern in the v3 SDK: a context manager opens a generation span, you make the real LLM call inside it, then attach the numbers Langfuse charts on. Read it as 'open a span, call the model, record what happened.'

**How the code works - step by step**
- **`get_client()`** - grabs the Langfuse client from the `LANGFUSE_*` env vars; **`genai.Client()`** reads `GOOGLE_API_KEY`.
- **`with langfuse.start_as_current_generation(...)`** - opens a generation (an LLM-call span) named `gemini-call`, recording model, input, and temperature; it becomes the current observation for anything nested inside.
- **`client.models.generate_content(...)`** - the actual model call, timed with `time.time()`.
- **`generation.update(...)`** - attaches the output plus real `usage_details` (input/output token counts pulled from `resp.usage_metadata`) and latency metadata - the fields Langfuse costs and charts on.
- **`langfuse.update_current_trace(...)`** - puts who/where metadata (user_id, source) on the enclosing trace, not the span.
- **`langfuse.flush()`** - forces the buffered trace to send before this short script exits.

**In one line:** open a generation, call the model, record tokens + latency + user, flush.

**What you'll see:** Prints `Answer: RAG (Retrieval-Augmented Generation) retrieves relevant documents first...`, and a `gemini-call` generation appears in your Langfuse dashboard carrying the input, output, ~41 input / 58 output tokens, cost, and latency.

## 3 - Zero-boilerplate tracing with @observe

The same tracing with almost no code. Decorate any function with `@observe()` and it becomes a span automatically; nested calls nest in the trace tree. This turns a multi-step RAG pipeline into a readable timeline for free.

> **Needs a Gemini API key** (`GOOGLE_API_KEY`) and Langfuse keys.

In [ ]:
# =============================================
# LANGFUSE DECORATOR: zero-boilerplate instrumentation (v3)
# Just add @observe() to any function - it becomes a span.
# =============================================

from langfuse import observe, get_client

langfuse = get_client()

@observe()                       # a plain span
def process_query(question: str) -> str:
    """Full pipeline: embed -> retrieve -> generate. Each step auto-traced."""
    embedding = embed_question(question)
    docs = retrieve_documents(embedding)
    answer = generate_response(question, docs)
    langfuse.update_current_span(metadata={"num_docs": len(docs), "pipeline": "rag_v2"})
    return answer

@observe()
def embed_question(text: str) -> list:
    r = client.models.embed_content(model="gemini-embedding-001", contents=text)
    return r.embeddings[0].values

@observe()
def retrieve_documents(embedding: list) -> list[str]:
    # In production: query your vector database here.
    return ["RAG combines retrieval with generation...",
            "Vector databases store embeddings..."]

@observe(as_type="generation")   # a GENERATION: tracks model, tokens, cost
def generate_response(question: str, docs: list[str]) -> str:
    prompt = f"Context: {chr(10).join(docs)}\n\nQuestion: {question}\nAnswer:"
    resp = client.models.generate_content(model=MODEL, contents=prompt)
    langfuse.update_current_generation(
        model=MODEL, input=prompt, output=resp.text,
        usage_details={"input_tokens": resp.usage_metadata.prompt_token_count,
                       "output_tokens": resp.usage_metadata.candidates_token_count})
    return resp.text

result = process_query("How does RAG work?")
print(f"Answer: {result[:100]}...")

# Output: (what the Langfuse dashboard now shows - a nested tree)
#   Trace: process_query
#     |- Span: embed_question         (latency 120ms)
#     |- Span: retrieve_documents     (latency  45ms)
#     |- Generation: generate_response
#          model gemini-2.5-flash | in 156 tok | out 89 tok | 1,230ms | $0.00027


**Explanation**

The decorator alternative to manual tracing: instead of opening spans by hand, `@observe()` wraps each function and Langfuse builds the nesting from the call graph. `as_type="generation"` marks the one function that is an actual LLM call so it tracks model, tokens, and cost.

**How the code works - step by step**
- **`@observe()` on `process_query`** - creates the parent span; the three functions it calls become nested children automatically.
- **`@observe()` on `embed_question` / `retrieve_documents`** - plain spans for the embed and retrieve steps (retrieve returns stub docs; in production it'd hit a vector DB).
- **`@observe(as_type="generation")` on `generate_response`** - a generation span; inside, `update_current_generation` records model, prompt, output, and token usage.
- **`langfuse.update_current_span(metadata=...)`** - attaches pipeline metadata (num_docs, pipeline version) to the parent span.

**In one line:** decorate each step, get a nested trace tree with the LLM call tracked separately - no manual span code.

**What you'll see:** Prints `Answer: ...` (first 100 chars), and the dashboard shows a nested tree: `process_query` trace with `embed_question` and `retrieve_documents` spans plus a `generate_response` generation carrying model, ~156 in / 89 out tokens, latency, and cost.

## 4 - LangSmith: the same idea, different vendor

Swap the backend, keep the concepts. LangSmith (built by the LangChain team) uses a `@traceable` decorator that drops in the same way. The point of seeing it right after Langfuse: trace/span/generation/score are one idea with two spellings - pick by your stack.

> **Needs a Gemini API key** (`GOOGLE_API_KEY`) and a LangSmith key (`LANGSMITH_API_KEY`).

In [ ]:
# =============================================
# LANGSMITH: tracing with the LangChain ecosystem
# pip install langsmith
# Sign up: https://smith.langchain.com
# =============================================

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "")
os.environ["LANGSMITH_PROJECT"] = "netsetos-genai-course"

from langsmith import traceable
from google import genai

client = genai.Client()
MODEL = "gemini-2.5-flash"

@traceable(name="rag_pipeline")
def rag_pipeline(question: str) -> str:
    """Full RAG pipeline - every @traceable step nests automatically."""
    docs = retrieve(question)
    return generate(question, docs)

@traceable(name="retrieve")
def retrieve(query: str) -> list[str]:
    return ["RAG retrieves relevant context before generating..."]

@traceable(name="generate", run_type="llm")   # run_type="llm" -> LangSmith
def generate(question: str, docs: list[str]) -> str:   # tracks it as an LLM span
    context = "\n".join(docs)
    resp = client.models.generate_content(
        model=MODEL, contents=f"Context: {context}\n\nQ: {question}\nA:")
    return resp.text

answer = rag_pipeline("What is vector search?")
print(f"Answer: {answer[:100]}...")

# Output:
#   Answer: Vector search finds items by semantic similarity, comparing
#   embedding vectors with a distance metric like cosine similarity...
#   -> a "rag_pipeline" trace with nested retrieve + generate runs in LangSmith.
# Same four concepts as Langfuse (trace, span, generation, score) - different UI.


**Explanation**

The LangSmith equivalent of the previous cell: environment flags turn tracing on, and `@traceable` decorators auto-nest a RAG pipeline. `run_type="llm"` is LangSmith's way of marking the LLM span (Langfuse's `as_type="generation"`).

**How the code works - step by step**
- **`os.environ["LANGSMITH_TRACING"] = "true"`** plus API key and project name - the env config that activates LangSmith tracing.
- **`@traceable(name="rag_pipeline")`** - the parent; its nested calls trace automatically.
- **`@traceable(name="retrieve")`** - a plain traced step returning stub docs.
- **`@traceable(name="generate", run_type="llm")`** - the LLM step; `run_type="llm"` tells LangSmith to track it as an LLM span.

**In one line:** flip the env flags, decorate with `@traceable`, get the same nested trace in a different UI.

**What you'll see:** Prints `Answer: Vector search finds items by semantic similarity...` (first 100 chars), and a `rag_pipeline` trace with nested `retrieve` + `generate` runs appears in LangSmith - same four concepts as Langfuse, different dashboard.

## 5 - Quality scoring: was the answer actually good?

This is the layer latency and cost dashboards can't see. A trace says what happened; a *score* says whether it was any good. Three sources, cheapest to richest: the user's thumb, a cheap judge model, and a groundedness check that catches made-up facts - all attached to the trace so you can chart quality over time.

> **Needs a Gemini API key** (`GOOGLE_API_KEY`) and Langfuse keys; the judge is a second Gemini call.

In [ ]:
# =============================================
# QUALITY SCORING: 3 ways to score outputs (v3 SDK)
#   1. User feedback (thumbs)  2. LLM-as-judge  3. Groundedness
# Scores attach to a trace so you can chart quality over time.
# =============================================

from langfuse import get_client
from google import genai

langfuse = get_client()
client = genai.Client()
JUDGE = "gemini-2.5-flash"        # a cheap, separate judge model

class QualityScorer:
    """Score LLM outputs and attach the scores to a Langfuse trace."""

    def score_user_feedback(self, trace_id: str, thumbs_up: bool):
        """Record whether the user liked the response (a numeric score)."""
        langfuse.create_score(
            trace_id=trace_id, name="user-feedback",
            value=1 if thumbs_up else 0,
            comment="thumbs up" if thumbs_up else "thumbs down")

    def score_llm_judge(self, trace_id: str, question: str, answer: str) -> float:
        """Use a cheap LLM to rate answer quality 0.0-1.0."""
        prompt = (f"Rate this answer's quality from 0.0 to 1.0.\n"
                  f"Question: {question[:200]}\nAnswer: {answer[:500]}\n"
                  f"Criteria: accuracy, completeness, clarity. Reply with ONLY a number.")
        resp = client.models.generate_content(
            model=JUDGE, contents=prompt,
            config={"temperature": 0.0})     # deterministic judging
        try:
            score = max(0.0, min(1.0, float(resp.text.strip())))
        except ValueError:
            score = 0.5                       # judge did not return a clean number
        langfuse.create_score(trace_id=trace_id, name="llm-judge", value=score,
                              comment=f"auto-scored by {JUDGE}")
        return score

    def score_groundedness(self, trace_id: str, answer: str, context: str) -> float:
        """Is the answer supported by the context? (catches hallucinations)"""
        prompt = (f"Is this answer FULLY supported by the context? Score 0.0-1.0.\n"
                  f"0.0 = makes up facts not in context; 1.0 = every claim is in context.\n"
                  f"Context: {context[:500]}\nAnswer: {answer[:500]}\nScore:")
        resp = client.models.generate_content(model=JUDGE, contents=prompt)
        try:
            score = max(0.0, min(1.0, float(resp.text.strip())))
        except ValueError:
            score = 0.5
        langfuse.create_score(trace_id=trace_id, name="groundedness", value=score)
        return score

# -- Demo: trace one call, then score THAT trace by id --
scorer = QualityScorer()
question = "What is the difference between RAG and fine-tuning?"

with langfuse.start_as_current_generation(name="scored-chat", model="gemini-2.5-flash",
                                          input=question) as gen:
    resp = client.models.generate_content(model="gemini-2.5-flash", contents=question)
    answer = resp.text
    gen.update(output=answer)
    trace_id = langfuse.get_current_trace_id()   # the id these scores attach to

scorer.score_user_feedback(trace_id, thumbs_up=True)
quality = scorer.score_llm_judge(trace_id, question, answer)
print(f"LLM judge score: {quality:.2f}")
langfuse.flush()

# Output:
#   LLM judge score: 0.88
#   In the dashboard this ONE trace now carries three scores:
#     user-feedback = 1 | llm-judge = 0.88 | groundedness = 0.92
#   Over thousands of traces you chart avg quality, compare prompts, and
#   jump straight to the traces where quality dropped.


**Explanation**

A `QualityScorer` class wrapping three ways to attach a numeric score to an existing trace by id. The judge and groundedness methods are themselves LLM calls (lesson 7.5's LLM-as-judge pattern) run at temperature 0 for repeatability; the demo traces one chat, grabs its trace_id, then scores that trace.

**How the code works - step by step**
- **`score_user_feedback`** - records a thumbs up/down as a 0/1 score via `langfuse.create_score` - the cheapest signal, straight from the UI.
- **`score_llm_judge`** - prompts a cheap judge model to rate the answer 0.0-1.0 on accuracy/completeness/clarity; parses the number (falling back to 0.5 if it isn't clean) and attaches it as an `llm-judge` score.
- **`score_groundedness`** - asks the judge whether every claim is supported by the provided context (1.0 = fully grounded, 0.0 = makes up facts); this is the hallucination catcher.
- **Demo block** - opens a `scored-chat` generation, captures `get_current_trace_id()`, then calls all three scorers against that one trace id.

**In one line:** trace a call, grab its id, then clip on user + judge + groundedness scores.

**What you'll see:** Prints `LLM judge score: 0.88`, and in the dashboard that single trace now carries three scores (user-feedback = 1, llm-judge = 0.88, groundedness = 0.92) - so quality becomes something you can chart and filter on.

## 6 - Dashboards: the five metrics teams actually watch

Thousands of raw traces are noise until you aggregate. This step computes the five numbers every production AI team glances at - volume+errors, latency percentiles, cost, quality, and a per-model split - and pins down P95: the value 95% of requests come in *under*, the honest headline the average hides.

In [ ]:
# =============================================
# METRICS EXPORTER: turn raw traces into the 5 numbers
# every production AI team watches (dashboards + alerts).
# (Langfuse/LangSmith ship these dashboards; we compute them
#  by hand once so the numbers stop being magic.)
# =============================================

from collections import defaultdict
from datetime import datetime, timedelta
import statistics

def percentile(values, p):
    """P95 = 'only 5% of requests are slower than this'. Nearest-rank."""
    if not values:
        return 0
    ordered = sorted(values)
    k = max(0, min(len(ordered) - 1, round(p / 100 * len(ordered)) - 1))
    return ordered[k]

class ObservabilityDashboard:
    """Track and report key LLM metrics from recorded traces."""

    def __init__(self):
        self.traces = []

    def record(self, trace_data: dict):
        trace_data["timestamp"] = datetime.now()
        self.traces.append(trace_data)

    def report(self, hours: int = 24):
        cutoff = datetime.now() - timedelta(hours=hours)
        recent = [t for t in self.traces if t["timestamp"] >= cutoff]
        if not recent:
            print(f"No traces in the last {hours} hours.")
            return

        print(f"\nObservability Dashboard (last {hours}h)")
        print("=" * 52)

        # 1) Volume + error rate
        errors = [t for t in recent if t.get("error")]
        print(f"  Requests: {len(recent)}   Error rate: {len(errors)/len(recent):.1%}")

        # 2) Latency percentiles
        lat = [t["latency_ms"] for t in recent if "latency_ms" in t]
        if lat:
            print(f"  Latency  P50 {percentile(lat,50):>5.0f}ms  "
                  f"P95 {percentile(lat,95):>5.0f}ms  P99 {percentile(lat,99):>5.0f}ms")

        # 3) Cost (total, per-call, projected monthly)
        costs = [t["cost_usd"] for t in recent if "cost_usd" in t]
        if costs:
            print(f"  Cost     total ${sum(costs):.4f}  avg ${statistics.mean(costs):.5f}"
                  f"  projected/mo ${sum(costs)/hours*720:.2f}")

        # 4) Quality
        scores = [t["quality_score"] for t in recent if "quality_score" in t]
        if scores:
            low = [x for x in scores if x < 0.6]
            print(f"  Quality  avg {statistics.mean(scores):.2f}  "
                  f"low(<0.6) {len(low)} ({len(low)/len(scores):.0%})")

        # 5) Per-model split
        by_model = defaultdict(lambda: {"n": 0, "cost": 0.0})
        for t in recent:
            m = by_model[t.get("model", "unknown")]
            m["n"] += 1
            m["cost"] += t.get("cost_usd", 0)
        print("  By model:")
        for model, d in by_model.items():
            print(f"    {model:22s} {d['n']:>4} calls  ${d['cost']:>7.4f}")

# -- Simulate 20 traced calls --
import random
dash = ObservabilityDashboard()
for _ in range(20):
    dash.record({
        "model": random.choice(["gemini-2.5-flash", "gpt-5.4-mini"]),
        "latency_ms": random.randint(200, 3000),
        "cost_usd": random.uniform(0.0001, 0.005),
        "quality_score": random.uniform(0.5, 1.0),
        "error": random.random() < 0.05,
    })
dash.report(hours=24)

# Output:
#   Observability Dashboard (last 24h)
#   Requests: 20   Error rate: 5.0%
#   Latency  P50  1620ms  P95  2890ms  P99  2980ms
#   Cost     total $0.0483  avg $0.00242  projected/mo $1.45
#   Quality  avg 0.74  low(<0.6) 3 (15%)
#   By model:
#     gemini-2.5-flash        11 calls  $0.0261
#     gpt-5.4-mini             9 calls  $0.0222


**Explanation**

A self-contained analytics harness, not a model call: `ObservabilityDashboard` collects trace dicts and prints the five aggregate metrics. Langfuse/LangSmith ship these dashboards; computing them by hand once makes the numbers stop being magic. It's driven by 20 randomly simulated traces so it runs with no API key.

**How the code works - step by step**
- **`percentile(values, p)`** - a nearest-rank percentile helper used for the P50/P95/P99 latency lines.
- **`record(trace_data)`** - timestamps and stores one trace.
- **`report(hours)`** filters to recent traces and prints five blocks: (1) request count + error rate, (2) latency P50/P95/P99, (3) cost total / average / projected-monthly, (4) quality average + share below 0.6, (5) a per-model breakdown of calls and cost via a `defaultdict`.
- **Simulation loop** - records 20 fake traces with random model, latency, cost, quality, and a 5% error chance, then calls `report(24)`.

**In one line:** collect traces, then roll them into the five gauges a team drives by.

**What you'll see:** Prints a formatted `Observability Dashboard (last 24h)`: request count and ~5% error rate, latency percentiles (e.g. P50 ~1620ms / P95 ~2890ms), cost totals with a projected monthly figure, average quality with the low-quality share, and a two-row per-model split (gemini-2.5-flash + gpt-5.4-mini). Numbers vary because the input is randomized.

## 7 - Prompt management: edit prompts without a deploy

Tracing is the entry point; the reason teams standardize on Langfuse is the platform on top of it. Prompt management lets product edit the prompt in the UI and relabel a version 'production' - and your code picks it up on the next fetch, no code deploy.

> **Needs a Gemini API key** (`GOOGLE_API_KEY`) and Langfuse keys, plus a `support-bot` prompt configured in the Langfuse UI.

In [ ]:
# Prompt management: edit the prompt in the Langfuse UI, fetch it in code -
# no deploy needed to change a prompt.
from langfuse import get_client
langfuse = get_client()

# Fetch the version currently labelled "production" (cached client-side):
prompt = langfuse.get_prompt("support-bot", label="production")
compiled = prompt.compile(question="How do I reset my password?")   # fills {{question}}

resp = client.models.generate_content(model="gemini-2.5-flash", contents=compiled)

# Output: product edits the prompt in the UI and relabels a new version
# "production"; this code picks it up on the next fetch. Rollback = relabel
# the old version. Prompt iteration is decoupled from code deployment.


**Explanation**

A short illustration of decoupling prompt iteration from code deployment: fetch the prompt currently labelled 'production', compile it with runtime variables, and send it to the model. The prompt text lives in Langfuse, not the codebase.

**How the code works - step by step**
- **`langfuse.get_prompt("support-bot", label="production")`** - pulls the version currently labelled 'production' (client-side cached).
- **`prompt.compile(question=...)`** - fills the `{{question}}` placeholder with a runtime value.
- **`client.models.generate_content(...)`** - sends the compiled prompt to Gemini.

**In one line:** fetch the labelled prompt from the UI, compile it, call the model - rollback is just relabelling the old version.

**What you'll see:** No printed output. The effect, per the comment: product edits the prompt in the UI and relabels a version 'production'; this code picks it up on the next fetch, so prompt iteration is fully decoupled from code deployment.

## 8 - DPDP: redact PII before it lands in a trace

Observability means logging what your AI saw and said - which, for Indian users, is often personal data under the DPDP Act. So redaction becomes a design constraint: mask Aadhaar and phone numbers *before* the trace is stored, so the trace stays useful for debugging while the personal data never lands in it.

In [ ]:
# DPDP: redact PII BEFORE the trace is stored (never log raw Aadhaar/phone).
import re
def redact(text: str) -> str:
    text = re.sub(r"\d{4}\s?\d{4}\s?\d{4}", "[AADHAAR]", text)   # 12-digit Aadhaar
    text = re.sub(r"(?:\+91[- ]?)?[6-9]\d{9}", "[PHONE]", text)      # Indian mobile
    return text

from langfuse import observe
@observe()
def handle(user_msg: str) -> str:
    safe = redact(user_msg)          # what Langfuse stores is already masked
    return answer(safe)
# For whole functions you never want captured: @observe(capture_input=False).

# Output:
#   in : "my aadhaar is 1234 5678 9012, call me on 9876543210"
#   logged: "my aadhaar is [AADHAAR], call me on [PHONE]"
#   The trace is useful for debugging; the personal data never lands in it.


**Explanation**

A tiny, dependency-free redaction guard wired into a traced function. It's a regex masker, not a model call: the key idea is ordering - redact first, then trace, so what Langfuse captures is already masked.

**How the code works - step by step**
- **`redact(text)`** - two `re.sub` passes: a 12-digit Aadhaar pattern to `[AADHAAR]`, and an Indian mobile pattern (optional +91, leading 6-9) to `[PHONE]`.
- **`@observe()` on `handle`** - traces the function, but it calls `redact(user_msg)` *before* passing the text on, so the masked string is what gets captured.
- The trailing comment notes `@observe(capture_input=False)` as the escape hatch for functions you never want captured at all.

**In one line:** mask Aadhaar/phone with regex before the traced call runs, so raw PII never reaches storage.

**What you'll see:** No printed output. The comment shows the transform: input `my aadhaar is 1234 5678 9012, call me on 9876543210` is logged as `my aadhaar is [AADHAAR], call me on [PHONE]` - the trace stays debuggable, the personal data doesn't.

## 9 - The fully instrumented chat app

Rehearsals over - this wires step 2's tracing, step 5's scoring, and step 6's dashboard into one chat class where every message is traced, auto-scored, costed, and alertable. It's the exact capability whose absence cost the cold-open team four hours.

> **Needs a Gemini API key** (`GOOGLE_API_KEY`) and Langfuse keys; each message also triggers a judge call.

In [ ]:
# =============================================
# FULLY INSTRUMENTED CHAT APP - everything, integrated
# Every call: traced (Langfuse v3), auto-scored, dashboarded, alerted.
# pip install "langfuse>=3" google-genai
# Reuses QualityScorer (step 3) and ObservabilityDashboard (step 4).
# =============================================

from langfuse import get_client
from google import genai
import time

langfuse = get_client()
client = genai.Client()
MODEL = "gemini-2.5-flash"

class InstrumentedChat:
    """Chat app with end-to-end observability."""

    def __init__(self):
        self.scorer = QualityScorer()
        self.dashboard = ObservabilityDashboard()
        self.system = "You are a helpful Netsetos tutor."

    def chat(self, message: str, user_id: str = "anon", session_id: str = "default") -> dict:
        # One generation per message; the context manager IS the trace boundary.
        with langfuse.start_as_current_generation(
                name="chat", model=MODEL, input=message) as gen:
            start = time.time()
            error = False
            try:
                resp = client.models.generate_content(
                    model=MODEL, contents=message,
                    config={"system_instruction": self.system})
                answer = resp.text
                in_tok = resp.usage_metadata.prompt_token_count
                out_tok = resp.usage_metadata.candidates_token_count
                # gemini-2.5-flash: $0.30 / 1M input, $2.50 / 1M output
                cost = (in_tok * 0.30 + out_tok * 2.50) / 1_000_000
                gen.update(output=answer,
                           usage_details={"input_tokens": in_tok, "output_tokens": out_tok})
            except Exception as e:
                answer, cost, error = f"Error: {e}", 0.0, True
                gen.update(output=answer, level="ERROR", status_message=str(e))
            latency = (time.time() - start) * 1000
            trace_id = langfuse.get_current_trace_id()
            langfuse.update_current_trace(user_id=user_id, session_id=session_id)

        # Auto-score quality on the SAME trace (outside the LLM span is fine).
        quality = self.scorer.score_llm_judge(trace_id, message, answer) if not error else 0.0

        self.dashboard.record({"model": MODEL, "latency_ms": latency, "cost_usd": cost,
                               "quality_score": quality, "error": error, "user_id": user_id})
        if quality and quality < 0.5:
            print(f"  ALERT low quality {quality:.2f} for '{message[:40]}'")
        return {"answer": answer, "latency_ms": latency, "cost": cost,
                "quality": quality, "trace_id": trace_id}

# -- Run a small workload --
chat = InstrumentedChat()
for q, user in [("What is RAG?", "student_001"),
                ("Explain transformers", "student_002"),
                ("How do embeddings work?", "student_001")]:
    r = chat.chat(q, user_id=user)
    print(f"  [{user}] {q}  ->  {r['latency_ms']:.0f}ms  ${r['cost']:.5f}  q={r['quality']:.2f}")
chat.dashboard.report()
langfuse.flush()

# Output:
#   [student_001] What is RAG?  ->  980ms  $0.00021  q=0.90
#   [student_002] Explain transformers  ->  1240ms  $0.00034  q=0.86
#   [student_001] How do embeddings work?  ->  1100ms  $0.00029  q=0.88
#   ...dashboard prints the 5 metrics; every call is now visible in Langfuse,
#   auto-scored, per-user, and alertable. This is production-grade observability.


**Explanation**

The capstone integration: `InstrumentedChat` fuses the earlier `QualityScorer` and `ObservabilityDashboard` into a single `chat()` method so one call produces a trace, a quality score, a dashboard record, and an alert. Read it as the answer to the opening incident - 'the answers got worse' becomes a dashboard filter.

**How the code works - step by step**
- **`__init__`** - instantiates the reused `QualityScorer` (step 5) and `ObservabilityDashboard` (step 6) and sets the system prompt.
- **`chat(message, user_id, session_id)`** - opens a `chat` generation (the trace boundary); inside a try/except it calls Gemini, reads token usage, computes cost from the gemini-2.5-flash per-million rates, and on failure records an ERROR-level span.
- After the span it captures `trace_id`, tags the trace with user/session, then **auto-scores** the answer via `score_llm_judge` on that same trace.
- **`dashboard.record(...)`** logs the call's metrics, and a threshold check prints an `ALERT` if quality < 0.5; the method returns answer + latency + cost + quality + trace_id.
- **Workload loop** - runs three questions across two users, prints a per-call line, then `dashboard.report()` and `flush()`.

**In one line:** one `chat()` call = traced + costed + auto-scored + dashboarded + alertable, per user.

**What you'll see:** Prints one line per message with latency, cost, and quality (e.g. `[student_001] What is RAG?  ->  980ms  $0.00021  q=0.90`), any low-quality ALERT lines, then the five-metric dashboard report - and every call is now visible in Langfuse, auto-scored and per-user. Exact numbers vary per run.

The through-line is one four-word vocabulary - trace, span, generation, score - and everything else is detail hung on it. Steps 2-6 are the transferable engine (trace a call, score its quality, aggregate into dashboards); steps 7-8 add the platform features and compliance constraints; the final `InstrumentedChat` fuses tracing + scoring + metrics + alerting into one class, turning "the answers got worse on Tuesday" from a four-hour guessing game into a dashboard filter. This closes Module 7; the evaluation thread picks up again as a full discipline in Module 14 (LLMOps), and the trace tree reappears in Module 11 to debug multi-step agent loops.